# Project domain tree

Recreates the project-domain diagram (Projects → Astronomy, Environment, Biology, Medical, Other Projects → the subcategories of the selected domain) as a selectable figure. Pick a domain and the diagram colors it; pick a subcategory as well and the domain opens up into its subcategories with only the path to the chosen one in color. Everything else is greyed out and the other domains stay closed.

Run the cells top to bottom: the first sets the color theme, the second holds the domain list, the third defines the chart, and the last one shows the selector.

In [ ]:
# Theme: pick the color family used by the chart (pumpkin, apricot, apple, teal, blueberry, eggplant, dragonfruit)
COLOR_FAMILY = "pumpkin"
PALETTE_FILE = "colorpalette.json"  # path to the palette file, relative to the notebook
OUTPUT_DIR = "output"  # folder the chart PNGs are saved to, relative to the notebook

import json
from pathlib import Path

import matplotlib.pyplot as plt

with open(Path(PALETTE_FILE)) as f:
    _data = json.load(f)
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(exist_ok=True)

palette = _data["palettes"]

families = palette["Color families"]
neutrals = palette["Neutrals"]


def shade(name, family=None):
    """Hex of one shade (lighter, light, base, dark, darker) of a color family."""
    family = family or COLOR_FAMILY
    return families["families"][family][families["shades"].index(name)]


def neutral(role):
    """Hex of a neutral role (background, gridline, axis, text, ...) via the grey ramp."""
    return neutrals["ramp"][neutrals["roles"][role]]


plt.rcParams["font.family"] = "sans-serif"
print(f"Theme: {COLOR_FAMILY}", {s: shade(s) for s in families["shades"]})

## Domains and subcategories

Five domains, each with its subcategories. A subcategory is a short label (the box in the diagram and the dropdown entry) mapped to a longer description (shown in the card below the diagram). Only the selected domain is opened up in the diagram; up to seven subcategories fit in one row, more are laid out in two rows.

Edit the dict here to change what can be selected.

In [ ]:
# INPUT: the domains and their subcategories, and the initial selection
# {domain: {short label: description}}; the order is the order in the diagram and the dropdowns.
DOMAINS = {
    "Astronomy": {
        "Galaxy classification": "Galaxy classification and astronomical image analysis",
        "Stars and planets": "Star and planet observation",
        "Supernovae and transients": "Supernova and transient-event detection",
        "Meteors and meteorites": "Meteor and meteorite observations",
        "Solar activity": "Solar activity and space-weather observations",
        "Planetary and lunar science": "Planetary science and lunar observation",
        "Archival data": "Identification of astronomical objects in archival data",
    },
    "Environment": {
        "Environmental monitoring": "air quality, noise pollution, water quality, soil quality, chemical contamination and environmental exposure",
        "Climate": "Climate monitoring and climate-change observations",
        "Weather": "Weather observations and meteorological monitoring",
        "Urban heat": "Urban microclimate and urban heat monitoring",
        "Biodiversity and ecosystems": "Biodiversity and ecosystem monitoring outside projects focused primarily on specific organisms",
        "Marine and freshwater": "Marine and freshwater environmental monitoring",
        "Forest and land use": "Forest, land-use and land-cover monitoring",
        "Pollution": "Pollution detection and reporting",
        "Natural hazards": "floods, droughts, wildfires, landslides and coastal erosion",
        "Ecosystem degradation": "Environmental change and ecosystem degradation",
        "Waste and litter": "Waste monitoring and litter mapping",
        "Light pollution": "Light pollution monitoring",
        "Environmental justice": "Environmental justice and community-based environmental assessment",
        "Agricultural and urban conditions": "Agricultural and urban environmental conditions",
    },
    "Biology": {
        "Plants": "plant identification, vegetation monitoring, phenology, invasive plants and plant diseases",
        "Avian": "bird observation, distribution, migration, nesting and population monitoring",
        "Marine": "marine organisms, marine biodiversity, coral reefs, marine mammals and other marine life",
        "Terrestrial invertebrates": "insects, pollinators, spiders, worms and other terrestrial invertebrates",
        "Amphibians and reptiles": "species distribution, population monitoring, migration and habitat observations",
        "Mammals": "terrestrial mammal observations, population monitoring, behaviour and distribution",
        "Other animals": "fish, freshwater organisms and animal groups not covered by the preceding categories",
    },
    "Medical": {
        "Epidemiology": "Epidemiology and disease surveillance",
        "Symptom tracking": "Symptom tracking and patient-reported outcomes",
        "Patient-led research": "Patient-led and participatory health research",
        "Public health": "Public health and community health monitoring",
        "Infectious disease": "Infectious disease reporting",
        "Chronic disease": "Chronic disease research",
        "Mental health": "Mental health research",
        "Environmental health": "Environmental health and exposure assessment",
        "Biomedical imaging": "Biomedical image classification and analysis",
        "Genomics": "Genomics and genetic research",
        "Drug discovery": "Drug discovery and biomedical data analysis",
        "Health behaviour": "Health behaviour and lifestyle research",
        "Clinical research": "Clinical research participation and health-data contribution",
    },
    "Other Projects": {
        "Social sciences": "sociology, anthropology, political science, economics and participatory social research",
        "Humanities": "history, linguistics, literature, oral histories and cultural heritage",
        "Geography and mapping": "community mapping, volunteered geographic information, land-use mapping and participatory GIS",
        "Urban studies and planning": "mobility, transport, housing, urban infrastructure and community planning",
        "Agriculture and food systems": "crop monitoring, farmer-led experimentation, soil management and food-system research",
        "Engineering and technology": "open hardware, sensor development, DIY instrumentation and technology co-design",
        "Computer science": "distributed computing, machine-learning data annotation, software development and computational problem-solving",
        "Energy": "household energy consumption, renewable energy and energy-system monitoring",
        "Education": "science learning, science literacy and educational research",
        "Economics and business": "citizen-led economic data collection and participatory market research",
    },
}

# Environmental exposure assessment can sit under both Environment and Medical, depending on whether the focus is
# environmental conditions or their effects on human health.

SELECTED = ("Biology", "Avian")  # (domain, subcategory) the selector below starts with; subcategory None = domain only

## Chart

`plot_domain_tree(domain, subcategory)` draws the tree with the selected domain opened up into its subcategory boxes, the path to the selected subcategory in the theme color and everything else in grey. The other domains stay closed. The card below shows the description of the selected subcategory. With `subcategory=None` the domain itself is the highlighted end of the path: nothing opens up and there is no card.

In [ ]:
import textwrap

from matplotlib.patches import FancyBboxPatch

DOMAIN_LABELS = {"Other Projects": "Other\nProjects"}  # line breaks for narrow boxes


def _wcag_contrast(a, b):
    def lum(hex_):
        rgb = [int(hex_[i:i + 2], 16) / 255 for i in (1, 3, 5)]
        rgb = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in rgb]
        return 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
    hi, lo = sorted((lum(a), lum(b)), reverse=True)
    return (hi + 0.05) / (lo + 0.05)


def _text_on(fill, dark, light="#FFFFFF"):
    """Whichever of two text colors reads better on the given fill."""
    return dark if _wcag_contrast(fill, dark) >= _wcag_contrast(fill, light) else light


def plot_domain_tree(
    domain,
    subcategory,
    path_fill=None,        # fill of the selected path (Projects and the domain); default: theme "lighter"
    path_edge=None,        # outline and connectors of the selected path; default: theme "base"
    path_text=None,        # text on the selected path; default: theme "darker"
    leaf_fill=None,        # fill of the selected subcategory; default: theme "dark"
    leaf_edge=None,        # outline of the selected subcategory; default: theme "darker"
    grey_fill=None,        # greyed-out boxes; default: light grey
    grey_edge=None,        # outline and connectors of greyed-out boxes; default: mid grey
    grey_text=None,        # text of greyed-out boxes; default: neutral axis
    background="white",
    label_size=12,
    figsize=(14, 9.5),
    save_to=None,
):
    """Projects -> domains -> subcategories of the selected domain, with only the path to the selected one in color.

    domain: a key of DOMAINS. subcategory: one of its short labels, or None to select only the domain.
    """
    if domain not in DOMAINS:
        raise ValueError(f"domain must be one of {list(DOMAINS)}")
    if subcategory is not None and subcategory not in DOMAINS[domain]:
        raise ValueError(f"subcategory must be one of the {domain} subcategories")

    path_fill = path_fill or shade("lighter")
    path_edge = path_edge or shade("base")
    path_text = path_text or shade("darker")
    leaf_fill = leaf_fill or shade("dark")
    leaf_edge = leaf_edge or shade("darker")
    grey_fill = grey_fill or palette["Neutrals"]["ramp"]["grey-100"]
    grey_edge = grey_edge or palette["Neutrals"]["ramp"]["grey-300"]
    grey_text = grey_text or neutral("axis")
    leaf_text = _text_on(leaf_fill, shade("darker"))

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(background)
    ax.set_facecolor(background)
    width, height = 100, 100 * figsize[1] / figsize[0]  # data units match inches, so rounded corners stay round
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_aspect("equal")
    ax.axis("off")

    def box(cx, cy, w, h, text, fill, edge, color, lw=1.4, size=label_size, weight="normal", zorder=3):
        ax.add_patch(FancyBboxPatch((cx - w / 2, cy - h / 2), w, h, boxstyle="round,pad=0,rounding_size=1.1",
                                    facecolor=fill, edgecolor=edge, linewidth=lw, zorder=zorder))
        ax.text(cx, cy, text, ha="center", va="center", fontsize=size, color=color, fontweight=weight,
                linespacing=1.15, zorder=zorder + 1)

    def connect(x0, y0, x1, y1, active):
        """Elbow from (x0, y0) down to (x1, y1) with the horizontal run halfway."""
        ym = (y0 + y1) / 2
        ax.plot([x0, x0, x1, x1], [y0, ym, ym, y1], color=path_edge if active else grey_edge,
                linewidth=2.2 if active else 1.4, solid_capstyle="round", solid_joinstyle="round",
                zorder=2 if active else 1)

    domains = list(DOMAINS)
    y_root, y_dom, y_row1, row_gap, y_card = height - 6, height - 19, height - 35, 10, 9
    dom_xs = {d: 12 + 19 * i for i, d in enumerate(domains)}
    dom_w, dom_h, sub_w, sub_h, sub_gap = 16.4, 6.4, 12.4, 7.6, 13.6

    # the selected domain opens up: up to seven subcategories in one row, more in two rows inside a panel
    subs = list(DOMAINS[domain]) if subcategory is not None else []
    rows = 1 if len(subs) <= 7 else 2
    cols = -(-len(subs) // rows)
    sub_pos = {}
    for i, name in enumerate(subs):
        r, c = divmod(i, cols)
        n_in_row = min(cols, len(subs) - r * cols)
        sub_pos[name] = (50 + (c - (n_in_row - 1) / 2) * sub_gap, y_row1 - r * row_gap)

    # connectors first so boxes sit on top; the selected path is drawn last among them
    for d in domains:
        connect(50, y_root - 2.6, dom_xs[d], y_dom + dom_h / 2, d == domain)
    if rows == 1:
        for name, (x, y) in sub_pos.items():
            connect(dom_xs[domain], y_dom - dom_h / 2, x, y + sub_h / 2, name == subcategory)
    else:
        pad = 1.8
        panel_w = (cols - 1) * sub_gap + sub_w + 2 * pad
        panel_top = y_row1 + sub_h / 2 + pad
        panel_bottom = y_row1 - row_gap - sub_h / 2 - pad
        connect(dom_xs[domain], y_dom - dom_h / 2, 50, panel_top, True)
        ax.add_patch(FancyBboxPatch((50 - panel_w / 2, panel_bottom), panel_w, panel_top - panel_bottom,
                                    boxstyle="round,pad=0,rounding_size=1.1", facecolor=background,
                                    edgecolor=path_edge, linewidth=1.8, zorder=1))

    box(50, y_root, 17, 5.2, "Projects", path_fill, path_edge, path_text, lw=1.8, size=label_size + 3, weight="bold")
    for d in domains:
        on = d == domain
        end = on and subcategory is None  # the domain itself ends the path
        fill, edge, color = (leaf_fill, leaf_edge, leaf_text) if end else (path_fill, path_edge, path_text) if on else (grey_fill, grey_edge, grey_text)
        box(dom_xs[d], y_dom, dom_w, dom_h, DOMAIN_LABELS.get(d, d), fill, edge, color,
            lw=1.8 if on else 1.4, size=label_size + 1, weight="bold" if on else "normal")
    for name, (x, y) in sub_pos.items():
        on = name == subcategory
        box(x, y, sub_w, sub_h, "\n".join(textwrap.wrap(name, 16)),
            leaf_fill if on else grey_fill, leaf_edge if on else grey_edge, leaf_text if on else grey_text,
            lw=1.8 if on else 1.4, size=label_size - 1, weight="bold" if on else "normal")

    # card with the description of the selected subcategory
    if subcategory is not None:
        body = DOMAINS[domain][subcategory]
        body = body[0].upper() + body[1:]
        card_w, card_h = 78, 10
        ax.add_patch(FancyBboxPatch((50 - card_w / 2, y_card - card_h / 2), card_w, card_h, boxstyle="round,pad=0,rounding_size=1.1",
                                    facecolor=path_fill, edgecolor=path_edge, linewidth=1.8, zorder=3))
        ax.text(50, y_card + card_h / 2 - 1.9, f"{domain}  \u203a  {subcategory}", ha="center", va="top",
                fontsize=label_size - 1, color=path_text, fontweight="bold", zorder=4)
        ax.text(50, y_card - 1.0, "\n".join(textwrap.wrap(body, 100)), ha="center", va="center", fontsize=label_size,
                color=path_text, linespacing=1.3, zorder=4)

    if subcategory is None:  # nothing opens up below the domains: crop the empty space
        bottom = y_dom - dom_h / 2 - 5
        ax.set_ylim(bottom, height)
        fig.set_size_inches(figsize[0], figsize[1] * (height - bottom) / height)

    fig.tight_layout()
    if save_to:
        fig.savefig(save_to, dpi=200, facecolor=fig.get_facecolor(), bbox_inches="tight")
    return fig, ax


fig, ax = plot_domain_tree(*SELECTED, save_to=OUTPUT_PATH / "domain_tree.png")
plt.show()

## Selector

Choose a domain; with the subcategory left on “none” only the domain is highlighted. Choose a subcategory as well and the domain opens up in the diagram, the others close, and the card shows the description. The PNG in the output folder is updated on every change. Needs a live Jupyter kernel (JupyterLab or the Jupyter extension of an editor), because it uses `ipywidgets`; headless runs just show the placeholder text.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

domain_menu = widgets.Dropdown(options=list(DOMAINS), value=SELECTED[0], description="Domain:",
                               style={"description_width": "80px"}, layout=widgets.Layout(width="320px"))
NO_SUBCATEGORY = ("(none)", None)


def _subcategory_options(domain):
    return [NO_SUBCATEGORY] + [(name, name) for name in DOMAINS[domain]]


subcategory_menu = widgets.Dropdown(options=_subcategory_options(SELECTED[0]), value=SELECTED[1], description="Subcategory:",
                                    style={"description_width": "80px"}, layout=widgets.Layout(width="95%"))
chart_area = widgets.Output()


def _redraw(*_):
    with chart_area:
        clear_output(wait=True)
        fig, _ax = plot_domain_tree(domain_menu.value, subcategory_menu.value, save_to=OUTPUT_PATH / "domain_tree.png")
        display(fig)
        plt.close(fig)


def _domain_changed(change):
    subcategory_menu.unobserve(_redraw, names="value")
    subcategory_menu.options = _subcategory_options(change["new"])  # also resets the value to "(none)"
    subcategory_menu.observe(_redraw, names="value")
    _redraw()


domain_menu.observe(_domain_changed, names="value")
subcategory_menu.observe(_redraw, names="value")
display(widgets.VBox([domain_menu, subcategory_menu, chart_area]))
_redraw()